# AMPS I · Week 01 — HW0: Environment and LLM-assisted debugging
**Due:** before the Week 2 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Part A — Environment (20 pts)
**A1 (5).** Run the setup cells above. The `--info` output must show `dolfinx : 0.11.x`.

**A2 (15).** Complete the cell: create a 64×64 unit-square mesh with 4 ranks and print, *from rank 0 only*,
(i) the number of cells owned by each rank (gather them) and (ii) their sum, which must equal $2 \times 64^2 = 8192$.

In [ ]:
%%fenicsx -np 4
from mpi4py import MPI
from dolfinx import mesh
comm = MPI.COMM_WORLD
msh = mesh.create_unit_square(comm, 64, 64)
# TODO: number of cells owned by this rank → msh.topology.index_map(2).size_local
# TODO: gather to rank 0 (comm.gather) and print the list and its sum on rank 0 only

## Part B — Debugging an LLM-generated solver (50 pts)
The cell below was produced by an LLM for the Week 1 model problem
$-\Delta u = f$, $u=0$ on $x=0,1$, $\partial u/\partial n=0$ on $y=0,1$, $u_{ex}=\sin(\pi x)\cos(\pi y)$.
It contains **six defects**: some crash, some run silently with wrong results, and one is a misleading "verification".

**B1 (5).** Run it as is. Copy the error message.

In [ ]:
%%fenicsx -np 1
# EXPECTED-FAIL: this cell crashes on purpose (B1)
# ===================================================================
#  "LLM-generated" Poisson solver — HW0 Part B (contains planted bugs)
#  Problem: -Δu = f on (0,1)^2,  u = 0 on x=0 and x=1,
#           ∂u/∂n = 0 on y=0 and y=1,  u_ex = sin(πx) cos(πy)
# ===================================================================
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh, default_scalar_type
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
N = 32
msh = mesh.create_unit_square(comm, N, N)

# Function space: continuous piecewise-linear Lagrange elements
V = fem.FunctionSpace(msh, ("Lagrange", 1))

# Exact solution and source term
x = ufl.SpatialCoordinate(msh)
u_exact = ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])
f = -2 * ufl.pi**2 * ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])

# Variational problem
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = f * v * ufl.dx

# Dirichlet boundary condition u = 0 on the left and right walls
fdim = msh.topology.dim - 1
facets = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0.0))
dofs = fem.locate_dofs_topological(V, fdim, facets)
bc = fem.dirichletbc(default_scalar_type(0.0), dofs, V)
# (Neumann condition on y = 0, 1 is natural: nothing to do)

# Solve
problem = LinearProblem(a, L, bcs=[bc],
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
uh = problem.solve()

# Error
error_form = fem.form((uh - u_exact)**2 * ufl.dx)
L2_error = np.sqrt(fem.assemble_scalar(error_form))
print(f"L2 error: {L2_error:.4e}")

# Verification
if np.all(np.isfinite(uh.x.array)) and np.max(np.abs(uh.x.array)) < 10:
    print("✅ Solver verified: solution is finite and bounded.")

**B2 (10). Crashes.** Using the API reference sheet, find the defects that stop the script from running.
For each: line, what is wrong, the 0.11 fix.

> *your answer*

**B3 (15). Silent defects.** After fixing the crashes the script runs and prints "✅ Solver verified".
Find the **three** defects that make the result wrong and explain how you detected each one.
*(Hints: is the $L^2$ error small? Re-derive $f=-\Delta u_{ex}$. Which walls get the Dirichlet condition?
Run the cell with `-np 2`: does the error change?)*

> *your answer*

**B4 (5). Verification theater.** Why does the final `if` check prove nothing? Propose a check that would have caught the defects.

> *your answer*

**B5 (15).** Write the corrected solver below as a function `solve_poisson(N)` and run a convergence study for
$N = 8, 16, 32, 64$ with **`-np 2`**. Print a table of $L^2$ and $H^1$ errors and observed rates.
Expected: rates → 2 and → 1; $L^2$ error at $N=32$ ≈ $1.36\times10^{-3}$.

In [ ]:
%%fenicsx -np 2
# TODO: corrected solver + convergence table (runs with -np 2)

## Part C — Your own LLM-generated solver (30 pts)
Textbook Exercise 1.11: $\;-\nabla\cdot\big((2+\sin 2\pi x)\,\nabla u\big) = f$ in $(0,1)^2$, $\;u=0$ on $\partial\Omega$.

**C1 (5).** Derive the weak form by hand (Markdown + LaTeX below).

**C2 (10).** Choose a manufactured solution $u_{ex}$ that vanishes on $\partial\Omega$ and let UFL compute
$f = -\nabla\cdot(k\nabla u_{ex})$. Prompt an LLM (with the API sheet) to write the solver. Paste the **unchanged**
generated code in a Markdown code block, then your corrected version in the code cell.

**C3 (15).** Verify: convergence table for P1 and P2 with `-np 2`. State the expected rates and whether you observe them.

**C1 — weak form**

> *your derivation*

**C2 — generated code (unchanged)**

```python
# paste here
```

In [ ]:
%%fenicsx -np 2
# TODO: corrected solver for Exercise 1.11 with MMS + convergence table for P1 and P2 (-np 2)

**C3 — conclusion**

> *expected vs observed rates*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …